# 🔥 AI Knowledge & Retrieval Assistant (RAG-style Lite)

### A beginner/intermediate LangChain-powered system that:

* loads external documents/text
* chunks information
* creates embeddings
* retrieves relevant context
* answers user questions grounded in provided data



---

## **Question-Answer over Documents**

### Tool that allows us to query a product catalog for items of interest.

### This LLM will extract a piece of text from PDF files, an internal database, or any other directed file path to answer questions about said document and get whatever needed information.

In [ ]:
# Removing conflicting versions
!pip uninstall -y langchain langchain-core langchain-community langchain-openai

# Installing compatible versions
!pip install langchain==0.1.16 langchain-openai==0.1.3

Found existing installation: langchain 1.3.1
Uninstalling langchain-1.3.1:
  Successfully uninstalled langchain-1.3.1
Found existing installation: langchain-core 1.4.0
Uninstalling langchain-core-1.4.0:
  Successfully uninstalled langchain-core-1.4.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 817.7/817.7 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.1/303.1 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 3.8 MB/s eta 0:00:00
  Attempting uninstall: tenacity


In [ ]:
import openai
from openai import OpenAI
import os
from google.colab import userdata
import textwrap

# Get the OpenAI API key from Colab secrets
api_key = userdata.get('OPENAI_API_KEY')

# Initialize OpenAI client with the retrieved API key
client = OpenAI(api_key = api_key)

print("OpenAI client successfully initialized.")

OpenAI client successfully initialized.


In [ ]:
# Account for Deprecation of LLM model
import datetime
# Get the current Date
current_date = datetime.datetime.now().date()

# Define the Date after which the Model should be set to "gpt-3.5-turbo"
target_date = datetime.date(2024, 6, 12)

# Set the Model variable based on the current Date
if current_date > target_date:
    llm_model = "gpt-4.1"
else:
    llm_model = "gpt-3.5-turbo"

In [ ]:
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import CSVLoader
from langchain.vectorstores import DocArrayInMemorySearch
from IPython.display import display, Markdown
from langchain.llms import OpenAI

In [ ]:
file = 'premier_league_23-24.csv'
loader = CSVLoader(file_path = file)



---

### But LLMs can only inspect a few thousand words at a time. So, if we have very large documents, how can our LLM answer questions about everything in them?

### This is where embeddings and vector storage come into play.

In [ ]:
from langchain.indexes import VectorstoreIndexCreator

In [ ]:
!pip install docarray

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 9.0 MB/s eta 0:00:00


In [ ]:
from langchain_openai import OpenAIEmbeddings

# Initializing the Embedding Model with the API key
embeddings = OpenAIEmbeddings(openai_api_key = api_key) # = Every chunk of your Document is being sent to OpenAI’s servers to create Vector Embeddings

# Creating the Vector Store Index, passing the initialized Embedding Model
index = VectorstoreIndexCreator(
    vectorstore_cls = DocArrayInMemorySearch,
    embedding = embeddings
).from_loaders([loader])

In [ ]:
query = "Please list all the fixtures that took place \
within the month of December in 2023."

In [ ]:
# Let's see the Response

# Initializing the OpenAI LLM with the API key
llm = OpenAI(openai_api_key = api_key)

# Passing the initialized LLM to the Query method
response = index.query(query, llm = llm)
display(Markdown(response))


1. Chelsea vs. Crystal Palace (27/12/2023)
2. Man United vs. Aston Villa (26/12/2023)
3. Fulham vs. Arsenal (31/12/2023)
4. Luton vs. Chelsea (30/12/2023)



---

#### Step-by-Step

In [ ]:
from langchain.document_loaders import CSVLoader
loader = CSVLoader(file_path = file)

In [ ]:
docs = loader.load()

In [ ]:
# Chunking the Document

from langchain.text_splitter import RecursiveCharacterTextSplitter

# Initializing the Text Splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000, # Max Size of each Chunk
    chunk_overlap  = 100 # Overlap between Chunks to maintain Context
)

# Splitting the Loaded Documents into Chunks
chunked_docs = text_splitter.split_documents(docs)

print(f"Original documents: {len(docs)}")
print(f"Chunked documents: {len(chunked_docs)}")
print("First chunk:")
print(chunked_docs[0])

Original documents: 380
Chunked documents: 380
First chunk:
page_content='League Division: E0\nDate: 11/8/23\nTime: 20:00\nHome: Burnley\nAway: Man City\nHome Goals: 0\nAway Goals: 3\nResult: A\nHalf-Time Home Goals: 0\nHalf-Time Away Goals: 2\nHalf-Time Result: A\nReferee: C Pawson\nHome Shots: 6\nAway Shots: 17\nHome Shots on Target: 1\nAway Shots on Target: 8\nHome Fouls: 11\nAway Fouls: 8\nHome Corners: 6\nAway Corners: 5\nHome Yellow Cards: 0\nAway Yellow Cards: 0\nHome Red Card(s): 1\nAway Red Card(s): 0' metadata={'source': 'premier_league_23-24.csv', 'row': 0}


In [ ]:
# Creating Embeddings for each Chunk and Storing them in a `DocArrayInMemorySearch` Vector Store
# This process converts Text data into Numerical Vectors

from langchain.vectorstores import DocArrayInMemorySearch

# Creating a Vector Store from the Chunked Documents and Embeddings
vectorstore = DocArrayInMemorySearch.from_documents(
    chunked_docs,
    embeddings
)

print("Vector store created successfully from chunked documents.")

Vector store created successfully from chunked documents.


In [ ]:
embed = embeddings.embed_query("Which team ended up winning the most matches?")

In [ ]:
# We want to Create Embeddings for all pieces of text we just Loaded
# And also store them in a Vector Store

db = DocArrayInMemorySearch.from_documents( # This is how we store in a Vector Store
    docs,
    embeddings
)

In [ ]:
query = "Please suggest which Team is the best in the league based on this Season's data."

In [ ]:
docs = db.similarity_search(query)

In [ ]:
docs[0]

Document(page_content='League Division: E0\nDate: 6/11/23\nTime: 20:00\nHome: Tottenham\nAway: Chelsea\nHome Goals: 1\nAway Goals: 4\nResult: A\nHalf-Time Home Goals: 1\nHalf-Time Away Goals: 1\nHalf-Time Result: D\nReferee: M Oliver\nHome Shots: 8\nAway Shots: 17\nHome Shots on Target: 5\nAway Shots on Target: 8\nHome Fouls: 12\nAway Fouls: 21\nHome Corners: 1\nAway Corners: 6\nHome Yellow Cards: 1\nAway Yellow Cards: 5\nHome Red Card(s): 2\nAway Red Card(s): 0', metadata={'source': 'premier_league_23-24.csv', 'row': 109})



---

### Using all this^ to do Question-Answering over our own Documents

In [ ]:
# 1: Create a Retriever from this Vector Store

retriever = db.as_retriever()

# Retriever: A generic interface, that can be underpinned by any method, that takes in a query and returns documents

In [ ]:
# Chat API

llm = ChatOpenAI(temperature = 0.0, model = llm_model,
                 openai_api_key = api_key) # For Text-Generation and a Natural Language response

/usr/local/lib/python3.12/dist-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 0.3.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import ChatOpenAI`.
  warn_deprecated(


In [ ]:
# Joining all Documents/Page Content into a Single piece of Text

qdocs = "".join([docs[i].page_content for i in range(len(docs))])

In [ ]:
response = llm.call_as_llm(f"{qdocs} Question: Please list the top four teams in the \
Premier League for this season based on all the matches' data and results, \
and summarize each team's performance within 50 characters.")

In [ ]:
display(Markdown(response))

Based on the data you provided, only matches involving Chelsea, Tottenham, Wolves, and Crystal Palace are included. There is not enough data to determine the actual Premier League top four for the season, but I can rank these four teams based on the results given:

### 1. **Chelsea**
- **Performance:** Strong wins, some losses, consistent scoring.

### 2. **Tottenham**
- **Performance:** Mixed results, heavy loss to Chelsea.

### 3. **Wolves**
- **Performance:** Impressive away win at Chelsea.

### 4. **Crystal Palace**
- **Performance:** Struggled, only losses in provided data.

**Note:** This ranking is only based on the limited matches provided, not the full season.

In [ ]:
qa_stuff = RetrievalQA.from_chain_type(
    llm = llm,
    chain_type = "stuff", # "Stuff Method" (most common): Stuffs all the Data into the Prompt as Context to Pass to the Language Model - but some LLMs have a Context Length limit
    retriever = retriever,
    verbose = True
)

In [ ]:
query =  "Please list the Winning-est teams in a table \
in markdown and summarize each one's performance."

In [ ]:
response = qa_stuff.run(query)

/usr/local/lib/python3.12/dist-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 0.2.0. Use invoke instead.
  warn_deprecated(




> Entering new RetrievalQA chain...

> Finished chain.


In [ ]:
display(Markdown(response))

Based on the provided match data, here is a table of the teams with the most wins ("Winning-est teams") and a summary of each team's performance:

### Table of Winning-est Teams

| Team      | Matches Played | Wins | Losses | Draws |
|-----------|---------------|------|--------|-------|
| Chelsea   | 3             | 2    | 1      | 0     |
| West Ham  | 1             | 1    | 0      | 0     |
| Wolves    | 1             | 1    | 0      | 0     |
| Tottenham | 3             | 0    | 3      | 0     |

---

### Team Performance Summaries

#### **Chelsea**
- **Wins:** 2 (vs Tottenham 4-1 away, vs Tottenham 2-0 home)
- **Losses:** 1 (vs Wolves 2-4 home)
- **Draws:** 0
- **Summary:** Chelsea had the most wins in the provided data, defeating Tottenham both home and away. However, they suffered a home defeat to Wolves, showing some inconsistency, especially defensively at home.

#### **West Ham**
- **Wins:** 1 (vs Tottenham 2-1 away)
- **Losses:** 0
- **Draws:** 0
- **Summary:** West Ham played one match and won away at Tottenham, showing efficiency and resilience, especially coming from behind at halftime.

#### **Wolves**
- **Wins:** 1 (vs Chelsea 4-2 away)
- **Losses:** 0
- **Draws:** 0
- **Summary:** Wolves played one match and secured an impressive away win at Chelsea, scoring four goals and demonstrating strong attacking performance.

#### **Tottenham**
- **Wins:** 0
- **Losses:** 3 (vs Chelsea twice, vs West Ham)
- **Draws:** 0
- **Summary:** Tottenham lost all three matches in the data, struggling both at home and away, including a heavy home defeat to Chelsea and a loss despite leading at halftime against West Ham.

---

**Conclusion:**  
Chelsea is the "winning-est" team in the provided matches, with two wins out of three games. West Ham and Wolves each have one win from one match, while Tottenham did not win any of their matches.

In [ ]:
response = index.query(query, llm = llm)

In [ ]:
# Automating Document Ingestion
# Breaks down Documents, Converts the Text into Numerical representations, and Stores them for fast, Semantic search

index = VectorstoreIndexCreator(
    vectorstore_cls = DocArrayInMemorySearch,
    embedding = embeddings,
).from_loaders([loader])



---

### Evaluation

In [ ]:
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import CSVLoader
from langchain.indexes import VectorstoreIndexCreator
from langchain.vectorstores import DocArrayInMemorySearch

In [ ]:
file = 'premier_league_23-24.csv'
loader = CSVLoader(file_path = file)
data = loader.load()

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(openai_api_key = api_key)
index = VectorstoreIndexCreator(
    vectorstore_cls = DocArrayInMemorySearch,
    embedding = embeddings
).from_loaders([loader])

In [ ]:
llm = ChatOpenAI(temperature = 0.0, model = llm_model, openai_api_key = api_key)
qa = RetrievalQA.from_chain_type(
    llm = llm,
    chain_type = "stuff",
    retriever = index.vectorstore.as_retriever(),
    verbose = True,
    chain_type_kwargs = {
        "document_separator": "<<<<>>>>>"
    }
)

In [ ]:
# Test Datapoints

print(data[10], '\n')
print(data[11])

page_content="League Division: E0\nDate: 18/08/2023\nTime: 19:45\nHome: Nott'm Forest\nAway: Sheffield United\nHome Goals: 2\nAway Goals: 1\nResult: H\nHalf-Time Home Goals: 1\nHalf-Time Away Goals: 0\nHalf-Time Result: H\nReferee: P Bankes\nHome Shots: 16\nAway Shots: 7\nHome Shots on Target: 4\nAway Shots on Target: 3\nHome Fouls: 5\nAway Fouls: 9\nHome Corners: 6\nAway Corners: 7\nHome Yellow Cards: 2\nAway Yellow Cards: 3\nHome Red Card(s): 0\nAway Red Card(s): 0" metadata={'source': 'premier_league_23-24.csv', 'row': 10} 

page_content='League Division: E0\nDate: 19/08/2023\nTime: 15:00\nHome: Fulham\nAway: Brentford\nHome Goals: 0\nAway Goals: 3\nResult: A\nHalf-Time Home Goals: 0\nHalf-Time Away Goals: 1\nHalf-Time Result: A\nReferee: D Bond\nHome Shots: 10\nAway Shots: 17\nHome Shots on Target: 2\nAway Shots on Target: 8\nHome Fouls: 13\nAway Fouls: 12\nHome Corners: 5\nAway Corners: 5\nHome Yellow Cards: 2\nAway Yellow Cards: 2\nHome Red Card(s): 1\nAway Red Card(s): 0' metada

In [ ]:
# LLM-Generated Examples (Automating the Process)

from langchain.evaluation.qa import QAGenerateChain

In [ ]:
example_gen_chain = QAGenerateChain.from_llm(ChatOpenAI(model = llm_model, openai_api_key = api_key))

In [ ]:
new_examples = []

for t in data[:3]:
    response = example_gen_chain.invoke({"doc": t})
    new_examples.append(response)

print(new_examples)

[{'doc': Document(page_content='League Division: E0\nDate: 11/8/23\nTime: 20:00\nHome: Burnley\nAway: Man City\nHome Goals: 0\nAway Goals: 3\nResult: A\nHalf-Time Home Goals: 0\nHalf-Time Away Goals: 2\nHalf-Time Result: A\nReferee: C Pawson\nHome Shots: 6\nAway Shots: 17\nHome Shots on Target: 1\nAway Shots on Target: 8\nHome Fouls: 11\nAway Fouls: 8\nHome Corners: 6\nAway Corners: 5\nHome Yellow Cards: 0\nAway Yellow Cards: 0\nHome Red Card(s): 1\nAway Red Card(s): 0', metadata={'source': 'premier_league_23-24.csv', 'row': 0}), 'qa_pairs': {'query': 'According to the document, how many red cards did Burnley receive in their match against Man City on 11/8/23, and how does this compare to the number of red cards received by Man City in the same match?', 'answer': 'Burnley received 1 red card in their match against Man City on 11/8/23, while Man City received 0 red cards.'}}, {'doc': Document(page_content="League Division: E0\nDate: 12/8/23\nTime: 12:30\nHome: Arsenal\nAway: Nott'm Fore

In [ ]:
new_examples[0]

{'doc': Document(page_content='League Division: E0\nDate: 11/8/23\nTime: 20:00\nHome: Burnley\nAway: Man City\nHome Goals: 0\nAway Goals: 3\nResult: A\nHalf-Time Home Goals: 0\nHalf-Time Away Goals: 2\nHalf-Time Result: A\nReferee: C Pawson\nHome Shots: 6\nAway Shots: 17\nHome Shots on Target: 1\nAway Shots on Target: 8\nHome Fouls: 11\nAway Fouls: 8\nHome Corners: 6\nAway Corners: 5\nHome Yellow Cards: 0\nAway Yellow Cards: 0\nHome Red Card(s): 1\nAway Red Card(s): 0', metadata={'source': 'premier_league_23-24.csv', 'row': 0}),
 'qa_pairs': {'query': 'According to the document, how many red cards did Burnley receive in their match against Man City on 11/8/23, and how does this compare to the number of red cards received by Man City in the same match?',
  'answer': 'Burnley received 1 red card in their match against Man City on 11/8/23, while Man City received 0 red cards.'}}

In [ ]:
data[0]

Document(page_content='League Division: E0\nDate: 11/8/23\nTime: 20:00\nHome: Burnley\nAway: Man City\nHome Goals: 0\nAway Goals: 3\nResult: A\nHalf-Time Home Goals: 0\nHalf-Time Away Goals: 2\nHalf-Time Result: A\nReferee: C Pawson\nHome Shots: 6\nAway Shots: 17\nHome Shots on Target: 1\nAway Shots on Target: 8\nHome Fouls: 11\nAway Fouls: 8\nHome Corners: 6\nAway Corners: 5\nHome Yellow Cards: 0\nAway Yellow Cards: 0\nHome Red Card(s): 1\nAway Red Card(s): 0', metadata={'source': 'premier_league_23-24.csv', 'row': 0})

---

### **LLM-Assisted Evaluation**

In [ ]:
# Hard-coded Examples

examples = [
    {
        "query": "Are there any Teams that went Undefeated?",
        "answer": "No"
    },
    {
        "query": "Which Team won the most games?",
        "answer": "Man City"
    }
]

In [ ]:
# LLM-Generated Examples (Automating the Process)

from langchain.evaluation.qa import QAGenerateChain

In [ ]:
example_gen_chain = QAGenerateChain.from_llm(ChatOpenAI(model = llm_model, openai_api_key = api_key))

In [ ]:
new_examples = []

for t in data[:3]:
    response = example_gen_chain.invoke({"doc": t})
    new_examples.append(response)

print(new_examples)

[{'doc': Document(page_content='League Division: E0\nDate: 11/8/23\nTime: 20:00\nHome: Burnley\nAway: Man City\nHome Goals: 0\nAway Goals: 3\nResult: A\nHalf-Time Home Goals: 0\nHalf-Time Away Goals: 2\nHalf-Time Result: A\nReferee: C Pawson\nHome Shots: 6\nAway Shots: 17\nHome Shots on Target: 1\nAway Shots on Target: 8\nHome Fouls: 11\nAway Fouls: 8\nHome Corners: 6\nAway Corners: 5\nHome Yellow Cards: 0\nAway Yellow Cards: 0\nHome Red Card(s): 1\nAway Red Card(s): 0', metadata={'source': 'premier_league_23-24.csv', 'row': 0}), 'qa_pairs': {'query': 'According to the document, how many total shots on target were made by both teams combined during the Burnley vs. Man City match on 11/8/23, and which team had more shots on target?', 'answer': 'The total number of shots on target made by both teams combined was 9 (Burnley had 1 and Man City had 8). Man City had more shots on target than Burnley.'}}, {'doc': Document(page_content="League Division: E0\nDate: 12/8/23\nTime: 12:30\nHome: Ar

In [ ]:
new_examples[0]

{'doc': Document(page_content='League Division: E0\nDate: 11/8/23\nTime: 20:00\nHome: Burnley\nAway: Man City\nHome Goals: 0\nAway Goals: 3\nResult: A\nHalf-Time Home Goals: 0\nHalf-Time Away Goals: 2\nHalf-Time Result: A\nReferee: C Pawson\nHome Shots: 6\nAway Shots: 17\nHome Shots on Target: 1\nAway Shots on Target: 8\nHome Fouls: 11\nAway Fouls: 8\nHome Corners: 6\nAway Corners: 5\nHome Yellow Cards: 0\nAway Yellow Cards: 0\nHome Red Card(s): 1\nAway Red Card(s): 0', metadata={'source': 'premier_league_23-24.csv', 'row': 0}),
 'qa_pairs': {'query': 'According to the document, what was the final score and the half-time score of the match between Burnley and Man City on 11/8/23, and which team had a player sent off?',
  'answer': 'The final score was Burnley 0, Man City 3. The half-time score was Burnley 0, Man City 2. Burnley had a player sent off (1 red card), while Man City had no red cards.'}}

In [ ]:
data[0]

Document(page_content='League Division: E0\nDate: 11/8/23\nTime: 20:00\nHome: Burnley\nAway: Man City\nHome Goals: 0\nAway Goals: 3\nResult: A\nHalf-Time Home Goals: 0\nHalf-Time Away Goals: 2\nHalf-Time Result: A\nReferee: C Pawson\nHome Shots: 6\nAway Shots: 17\nHome Shots on Target: 1\nAway Shots on Target: 8\nHome Fouls: 11\nAway Fouls: 8\nHome Corners: 6\nAway Corners: 5\nHome Yellow Cards: 0\nAway Yellow Cards: 0\nHome Red Card(s): 1\nAway Red Card(s): 0', metadata={'source': 'premier_league_23-24.csv', 'row': 0})

### Combining Examples

### LLM-Assisted Evaluation

In [ ]:
# Adding these^ Examples to Examples we have already Created

examples += new_examples

# LLM-Assisted Evaluation

qa_examples = []
for ex in examples:
    if "query" in ex and "answer" in ex:
        qa_examples.append(ex)
    elif "qa_pairs" in ex and "query" in ex["qa_pairs"] and "answer" in ex["qa_pairs"]:
        qa_examples.append({"query": ex["qa_pairs"]["query"], "answer": ex["qa_pairs"]["answer"]})
    else:
        # Skip or handle examples with unexpected format if necessary
        pass

# Creating Predictions for all the Different Examples

predictions = qa.batch(qa_examples)



> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.


In [ ]:
from langchain.evaluation.qa import QAEvalChain

In [ ]:
llm = ChatOpenAI(temperature = 0, model = llm_model, openai_api_key = api_key)
eval_chain = QAEvalChain.from_llm(llm)

In [ ]:
graded_outputs = []

for example, prediction in zip(qa_examples, predictions):
    result = eval_chain.invoke({
        "query": example["query"],
        "answer": example["answer"],
        "result": prediction["result"]
    })

    graded_outputs.append(result)

print(graded_outputs)

[{'query': 'Are there any Teams that went Undefeated?', 'answer': 'No', 'result': 'Based on the provided match data, there is not enough information to determine if any team went completely undefeated throughout the season. The context only includes a few selected matches involving teams like Man United, Sheffield United, Aston Villa, Chelsea, and Tottenham, but not the full season results for any team.\n\nIf you have a specific team in mind or more complete data, I can help check their record. Otherwise, with the current information, I cannot confirm any undefeated teams.', 'results': 'CORRECT'}, {'query': 'Which Team won the most games?', 'answer': 'Man City', 'result': 'Based on the provided match results:\n\n1. Tottenham vs Chelsea (6/11/23): Chelsea won (Chelsea 4-1 Tottenham)\n2. Brighton vs Arsenal (6/4/24): Arsenal won (Arsenal 3-0 Brighton)\n3. Chelsea vs Wolves (4/2/24): Wolves won (Wolves 4-2 Chelsea)\n4. Chelsea vs Tottenham (2/5/24): Chelsea won (Chelsea 2-0 Tottenham)\n\n

In [ ]:
for i, eg in enumerate(examples):
    print(f"Example {i}:")
    print("Question: " + predictions[i]['query'])
    print("Real Answer: " + predictions[i]['answer'])
    print("Predicted Answer: " + predictions[i]['result'])
    print("Predicted Grade: " + graded_outputs[i]['results'])
    print()

Example 0:
Question: Are there any Teams that went Undefeated?
Real Answer: No
Predicted Answer: Based on the provided match data, there is not enough information to determine if any team went completely undefeated throughout the season. The context only includes a few selected matches involving teams like Man United, Sheffield United, Aston Villa, Chelsea, and Tottenham, but not the full season results for any team.

If you have a specific team in mind or more complete data, I can help check their record. Otherwise, with the current information, I cannot confirm any undefeated teams.
Predicted Grade: CORRECT

Example 1:
Question: Which Team won the most games?
Real Answer: Man City
Predicted Answer: Based on the provided match results:

1. Tottenham vs Chelsea (6/11/23): Chelsea won (Chelsea 4-1 Tottenham)
2. Brighton vs Arsenal (6/4/24): Arsenal won (Arsenal 3-0 Brighton)
3. Chelsea vs Wolves (4/2/24): Wolves won (Wolves 4-2 Chelsea)
4. Chelsea vs Tottenham (2/5/24): Chelsea won (Che



---

### **Agents**

In [ ]:
# Built-in LangChain tools

!pip install -U wikipedia

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=76420326b1c47e07831e8f2c1eb47d72c376356b58bb6bd01f97cf03a9dc2d4d
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia


In [ ]:
!pip uninstall -y langchain langchain-core langchain-openai langchain-community langchain-experimental openai

!pip install \
langchain==0.1.16 \
langchain-openai==0.1.3 openai \
langchain-experimental==0.0.56 \
openai==1.30.1

Found existing installation: langchain 0.1.16
Uninstalling langchain-0.1.16:
  Successfully uninstalled langchain-0.1.16
Found existing installation: langchain-core 0.1.53
Uninstalling langchain-core-0.1.53:
  Successfully uninstalled langchain-core-0.1.53
Found existing installation: langchain-openai 0.1.3
Uninstalling langchain-openai-0.1.3:
  Successfully uninstalled langchain-openai-0.1.3
Found existing installation: langchain-community 0.0.38
Uninstalling langchain-community-0.0.38:
  Successfully uninstalled langchain-community-0.0.38
Found existing installation: openai 1.109.1
Uninstalling openai-1.109.1:
  Successfully uninstalled openai-1.109.1
  Using cached langchain-0.1.16-py3-none-any.whl.metadata (13 kB)
  Using cached langchain_openai-0.1.3-py3-none-any.whl.metadata (2.5 kB)
  Using cached langchain_community-0.0.38-py3-none-any.whl.metadata (8.7 kB)
  Using cached langchain_core-0.1.53-py3-none-any.whl.metadata (5.9 kB)
Using cached langchain-0.1.16-py3-none-any.whl (81

In [ ]:
!pip install httpx==0.27.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 1.2 MB/s eta 0:00:00
  Attempting uninstall: httpx
    Found existing installation: httpx 0.28.1
    Uninstalling httpx-0.28.1:
      Successfully uninstalled httpx-0.28.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph 1.2.1 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.1.53 which is incompatible.
google-adk 1.29.0 requires tenacity<10.0.0,>=9.0.0, but you have tenacity 8.5.0 which is incompatible.
mcp 1.27.1 requires httpx<1.0.0,>=0.27.1, but you have httpx 0.27.0 which is incompatible.
google-genai 1.68.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.27.0 which is incompatible.
firebase-admin 6.9.0 requires httpx[http2]==0.28.1, but you have httpx 0.27.0 which is incompatible.
langgraph-checkpoint 4.1.0 requires langchain-core>=0.2.38, but you have langchain-core 0.1.

In [ ]:
from langchain.agents import AgentType
from langchain_experimental.agents.agent_toolkits import create_python_agent
from langchain_experimental.tools.python.tool import PythonREPLTool
from langchain_openai import ChatOpenAI

In [ ]:
import openai
from openai import OpenAI
import os
from google.colab import userdata
import textwrap

# Get the OpenAI API key from Colab secrets
api_key = userdata.get('OPENAI_API_KEY')

# Initialize OpenAI client with the retrieved API key
client = OpenAI(api_key = api_key)

print("OpenAI client successfully initialized.")

OpenAI client successfully initialized.


In [ ]:
# Account for Deprecation of LLM model
import datetime
# Get the current Date
current_date = datetime.datetime.now().date()

# Define the Date after which the Model should be set to "gpt-3.5-turbo"
target_date = datetime.date(2024, 6, 12)

# Set the Model variable based on the current Date
if current_date > target_date:
    llm_model = "gpt-4"
else:
    llm_model = "gpt-3.5-turbo"

In [ ]:
llm = ChatOpenAI(temperature = 0, model = llm_model, openai_api_key = api_key)

In [ ]:
from langchain.agents import load_tools

tools = load_tools(["llm-math", "wikipedia"], llm = llm)

In [ ]:
from langchain.agents import initialize_agent

agent = initialize_agent(
    tools,
    llm,
    agent = AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors = True,
    verbose = True)

In [ ]:
agent.run("What are the top 25%-performing teams of the Premier League from Season 2023-2024?")
langchain.debug = False



> Entering new AgentExecutor chain...
Thought: To answer this question, I need to know the total number of teams in the Premier League during the 2023-2024 season and the ranking of the teams. This information can be obtained from Wikipedia.
Action:
```
{
  "action": "wikipedia",
  "action_input": "2023–24 Premier League"
}
```
Observation: Page: 2023–24 Premier League
Summary: The 2023–24 Premier League was the 32nd season of the Premier League and the 125th season of top-flight English football overall. The season began on 11 August 2023 and concluded on 19 May 2024. 
Manchester City, the defending champions, won their fourth consecutive title, the first team to do so in English men's football.
This season was significant as it was affected by points deductions handed out to both Everton and Nottingham Forest, as part of the Premier League’s crackdown on financial breaches by clubs. Everton received two separate points deductions (a 10-point deduction, later reduced to six, in Nove

JSONDecodeError: Expecting value: line 1 column 1 (char 0)



---

### Defining my Own Tool

In [ ]:
!pip install DateTime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.7/270.7 kB 9.5 MB/s eta 0:00:00


In [ ]:
from langchain.agents import tool
from datetime import date

In [ ]:
# Making a Tool that tells us what the Current Date is

@tool # This is applied to the upcoming Function & Converts it to a Tool that the Agent can use
def time(text: str) -> str:
    """Returns todays date, use this for any \
    questions related to knowing todays date. \
    The input should always be an empty string, \
    and this function will always return todays \
    date - any date mathmatics should occur \
    outside this function."""
    return str(date.today())

In [ ]:
agent = initialize_agent(
    tools + [time],
    llm,
    agent = AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors = True,
    verbose = True)

# DISCLAIMER: Agents might sometimes come to the Wrong Conclusion
# If they do, Please Try Running them again

In [ ]:
agent.run("Who is the Premier League's current Assist Record-holder?")



> Entering new AgentExecutor chain...
Thought: I need to find out who currently holds the record for the most assists in the Premier League. I can use the wikipedia tool to find this information.
Action:
```
{
  "action": "wikipedia",
  "action_input": "Premier League records and statistics"
}
```

JSONDecodeError: Expecting value: line 1 column 1 (char 0)



---

### Memory

Trying ConversationBufferMemory this time.

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory

In [ ]:
from google.colab import userdata
api_key = userdata.get('OPENAI_API_KEY')

# Using LangChain to Manage a Chat/Chatbot Conversation

llm = ChatOpenAI(temperature = 0.0, model = llm_model,
                 openai_api_key = api_key) # Setting the LLM as the Chat Interface of OpenAI
memory = ConversationBufferMemory()
conversation = ConversationChain(
    llm = llm,
    memory = memory,
    verbose = True
)

In [ ]:
# Example
conversation.predict(input = "Hi, my friend is a huge Chelsea fan! Did they do well this season?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: Hi, my friend is a huge Chelsea fan! Did they do well this season?
AI:

> Finished chain.


'Hi! That’s awesome—Chelsea has a passionate fanbase. For the 2023/24 season, Chelsea had a bit of a rollercoaster ride. They finished 6th in the Premier League, which was an improvement compared to their 12th-place finish the previous season, but still below the high standards Chelsea fans are used to.\n\nUnder manager Mauricio Pochettino, the team showed flashes of exciting, attacking football, especially towards the end of the season. Young talents like Cole Palmer really stood out—Palmer was actually one of the league’s top scorers and a bright spot for the club. Chelsea also reached the Carabao Cup final but lost to Liverpool in extra time, and they made it to the FA Cup semi-finals before being knocked out by Manchester City.\n\nSo, while they didn’t win any trophies, there were definite signs of progress and reasons for optimism. If your friend is a Chelsea fan, they’ll probably be hoping for even better things next season!'

In [ ]:
print(memory.buffer)

Human: Hi, my friend is a huge Chelsea fan! Did they do well this season?
AI: Hi! That’s awesome—Chelsea has a passionate fanbase. For the 2023/24 season, Chelsea had a bit of a rollercoaster ride. They finished 6th in the Premier League, which was an improvement compared to their 12th-place finish the previous season, but still below the high standards Chelsea fans are used to.

Under manager Mauricio Pochettino, the team showed flashes of exciting, attacking football, especially towards the end of the season. Young talents like Cole Palmer really stood out—Palmer was actually one of the league’s top scorers and a bright spot for the club. Chelsea also reached the Carabao Cup final but lost to Liverpool in extra time, and they made it to the FA Cup semi-finals before being knocked out by Manchester City.

So, while they didn’t win any trophies, there were definite signs of progress and reasons for optimism. If your friend is a Chelsea fan, they’ll probably be hoping for even better th

In [ ]:
memory.load_memory_variables({})

{'history': 'Human: Hi, my friend is a huge Chelsea fan! Did they do well this season?\nAI: Hi! That’s awesome—Chelsea has a passionate fanbase. For the 2023/24 season, Chelsea had a bit of a rollercoaster ride. They finished 6th in the Premier League, which was an improvement compared to their 12th-place finish the previous season, but still below the high standards Chelsea fans are used to.\n\nUnder manager Mauricio Pochettino, the team showed flashes of exciting, attacking football, especially towards the end of the season. Young talents like Cole Palmer really stood out—Palmer was actually one of the league’s top scorers and a bright spot for the club. Chelsea also reached the Carabao Cup final but lost to Liverpool in extra time, and they made it to the FA Cup semi-finals before being knocked out by Manchester City.\n\nSo, while they didn’t win any trophies, there were definite signs of progress and reasons for optimism. If your friend is a Chelsea fan, they’ll probably be hoping 

In [ ]:
memory = ConversationBufferMemory() # This is how LangChain is Storing the Conversation

In [ ]:
memory.save_context({"input": "We also have a Tottenham fan within our friends circle."}, # Adding new things to the Memory
                    {"output": "Tottenham fan noted."})

In [ ]:
print(memory.buffer)

Human: We also have a Tottenham fan within our friends circle.
AI: Tottenham fan noted.


In [ ]:
memory.load_memory_variables({})

{'history': 'Human: We also have a Tottenham fan within our friends circle.\nAI: Tottenham fan noted.'}

In [ ]:
memory.save_context({"input": "We have a friendly rivalry between the Chelsea and Tottenham fans."},
                    {"output": "That's good!"})

In [ ]:
memory.load_memory_variables({})

{'history': "Human: We also have a Tottenham fan within our friends circle.\nAI: Tottenham fan noted.\nHuman: We have a friendly rivalry between the Chelsea and Tottenham fans.\nAI: That's good!"}



---

### Output Parser - Chained with the Wikipedia Tool

In [ ]:
# e.g., Extract Information from a Premier League fan's questions and Format that Output in a JSON Format

plfan_opinion = """\
I have been a Manchester United fan since I was a kid \
and when I properly began watching football. \

However, now I realize that those glory days were \
to be cherished the most, considering how we fallen \
off since Sir Alex's retirement. \

Thankfully, this end of the 2025-2026 season sort-of \
gives me hope, as we have qualified for the UCL and \
Captain Fantastic has also secured the PL assist record. \

Based on our Manchester United's matches from, \
do you think we can contend for the league title next \
season?
"""

review_template = """\
For the following text, extract the following information:

fan_loyalty: Which PL team does the fan support? \
Write down the Name of the Team which he expresses affinity for, if applicable, otherwise NA.

question_type: Is his question open-ended or of the fact-based single-answer kind? \
Answer: Open-Ended or Factual.

Format the output as JSON with the following keys:
fan_loyalty
question_type

text: {text}
"""

In [ ]:
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

# Define your desired output structure
class FanOpinion(BaseModel):
    fan_loyalty: str = Field(description="The Premier League team the fan supports or NA if not applicable.")
    question_type: str = Field(description="Type of question: 'Open-Ended' or 'Factual'.")

# Set up a parser + inject instructions into the prompt template
parser = PydanticOutputParser(pydantic_object=FanOpinion)


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(plfan_opinion)
print(prompt_template)

input_variables=[] messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], template="I have been a Manchester United fan since I was a kid and when I properly began watching football. \nHowever, now I realize that those glory days were to be cherished the most, considering how we fallen off since Sir Alex's retirement. \nThankfully, this end of the 2025-2026 season sort-of gives me hope, as we have qualified for the UCL and Captain Fantastic has also secured the PL assist record. \nBased on our Manchester United's matches from, do you think we can contend for the league title next season?\n"))]


In [ ]:
prompt_template_with_parser = ChatPromptTemplate.from_messages(
    [
        ("system", "Answer the user query as a JSON object, following the schema: {format_instructions}"),
        ("human", "{text}")
    ]
)

# Adding Parser Instructions to the Template
prompt = prompt_template_with_parser.partial(format_instructions=parser.get_format_instructions())

print(prompt)

input_variables=['text'] partial_variables={'format_instructions': 'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"fan_loyalty": {"title": "Fan Loyalty", "description": "The Premier League team the fan supports or NA if not applicable.", "type": "string"}, "question_type": {"title": "Question Type", "description": "Type of question: \'Open-Ended\' or \'Factual\'.", "type": "string"}}, "required": ["fan_loyalty", "question_type"]}\n```'} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['format_instructions'], template='Answer the user quer

In [ ]:
messages = prompt_template.format_messages(text = plfan_opinion)
chat = ChatOpenAI(temperature = 0.0, model = llm_model, openai_api_key = api_key) # Creating the OpenAI endpoint
response = chat.invoke(messages) # Calling the endpoint
print(response.content) # Printing the Response

As an AI, I don't have real-time data or the ability to predict future events. However, I can tell you that a team's success in the future often depends on various factors such as the quality of the players, the strategies of the coach, the team's health, and their overall performance in the previous season. If Manchester United has performed well in the 2025-2026 season, it could be a good indicator of their potential in the upcoming season. However, the unpredictability of football means nothing is guaranteed.


In [ ]:
formatted_prompt_for_llm = prompt.format(text=plfan_opinion)

# Invoke the LLM with the formatted prompt
llm_response = chat.invoke(formatted_prompt_for_llm)

# Parse the LLM's response
parsed_output = parser.parse(llm_response.content)

print("LLM's raw response:")
print(llm_response.content)
print("\nParsed JSON output:")
print(parsed_output)
print(f"Fan Loyalty: {parsed_output.fan_loyalty}")
print(f"Question Type: {parsed_output.question_type}")

LLM's raw response:
{"fan_loyalty": "Manchester United", "question_type": "Open-Ended"}

Parsed JSON output:
fan_loyalty='Manchester United' question_type='Open-Ended'
Fan Loyalty: Manchester United
Question Type: Open-Ended


Now, let's see how to get content from Wikipedia using the agent and then apply this parser.

In [ ]:
wikipedia_tool = next(tool for tool in tools if tool.name == 'wikipedia')

# Get some content using the Wikipedia tool with a more general query
wikipedia_query = "Premier League"
try:
    wikipedia_content = wikipedia_tool.run(wikipedia_query)
    print(f"Wikipedia content for '{wikipedia_query}':\n{wikipedia_content[:500]}...") # Print first 500 chars
except JSONDecodeError:
    print(f"Error: Failed to retrieve Wikipedia content for '{wikipedia_query}'. The Wikipedia API might be experiencing issues or returning non-JSON data.")
    wikipedia_content = "" # Set to empty string or handle as appropriate

Wikipedia content for 'Premier League':
Page: Premier League
Summary: The Premier League is a professional association football league in England and the highest level of the English football league system. Contested by 20 clubs, it operates on a system of promotion and relegation with the English Football League (EFL). Seasons usually run from August to May, with each team playing 38 matches: two against each other team, one home and one away. Most games are played on weekend afternoons, with occasional weekday evening fixtures.
The ...


In [ ]:
# Now, let's apply our output parser to this Wikipedia content
# We'll re-use the 'review_template' logic to extract information, imagining the Wikipedia content is a 'fan opinion' for demonstration.
# Note: The 'review_template' is designed for fan opinions, so the output might not be semantically meaningful for Wikipedia content, but it demonstrates the parsing process.

formatted_prompt_for_wiki_content = prompt.format(text=wikipedia_content[:1000]) # Using first 1000 chars of wikipedia content

llm_response_wiki = chat.invoke(formatted_prompt_for_wiki_content)

parsed_output_wiki = parser.parse(llm_response_wiki.content)

print("\nLLM's raw response for Wikipedia content:")
print(llm_response_wiki.content)
print("\nParsed JSON output for Wikipedia content:")
print(parsed_output_wiki)
print(f"Fan Loyalty (from Wiki content): {parsed_output_wiki.fan_loyalty}")
print(f"Question Type (from Wiki content): {parsed_output_wiki.question_type}")


LLM's raw response for Wikipedia content:
{"fan_loyalty": "NA", "question_type": "Open-Ended"}

Parsed JSON output for Wikipedia content:
fan_loyalty='NA' question_type='Open-Ended'
Fan Loyalty (from Wiki content): NA
Question Type (from Wiki content): Open-Ended
